# Diabetes Risk Analysis

Exploratory analysis of the bundled diabetes dataset. This notebook describes associations with the recorded `Outcome`; it does not train a model, predict future onset, establish causality, or support clinical decisions. Dataset provenance and redistribution terms remain to be verified.


## Data Collection & Loading
### Data Sources:
The dataset used in this project is a CSV file called diabetes.csv. It contains health data for 768 individuals, including factors like pregnancies, glucose levels, blood pressure, BMI, insulin levels, and whether the individual has diabetes (indicated by the 'Outcome' column)

### Loading the Data:
To load the data into the environment, follow these steps:

1. First, import the necessary libraries like pandas.
2. Use the pandas.read_csv() function to load the dataset from the CSV file.
3. After loading, review the dataset by calling df.head() to ensure it’s loaded correctly and check the first few rows.
4. Optionally, use df.info() to confirm the data types and check for any missing values or anomalies.

In [ ]:
# Load the data
import pandas as pd
data = pd.read_csv("diabetes.csv")
data.head()


In [ ]:
data.info()

## Data Description

### Data Structure:
- **Number of records (rows)**: 768
- **Number of features (columns)**: 9

### Variable Description:
| Variable Name               | Description                                                                 | Data Type  |
|-----------------------------|-----------------------------------------------------------------------------|------------|
| `Pregnancies`               | Number of times the patient has been pregnant                               | Numerical  |
| `Glucose`                   | Plasma glucose concentration (mg/dL) measured after 2 hours in an oral test | Numerical  |
| `BloodPressure`             | Diastolic blood pressure (mm Hg)                                            | Numerical  |
| `SkinThickness`             | Skinfold thickness (mm) measured on the triceps                             | Numerical  |
| `Insulin`                   | Serum insulin levels (mu U/ml)                                              | Numerical  |
| `BMI`                       | Body Mass Index (weight in kg/height in m²)                                 | Numerical  |
| `DiabetesPedigreeFunction`  | A score indicating the likelihood of diabetes based on family history       | Numerical  |
| `Age`                       | Age of the individual in years                                              | Numerical  |
| `Outcome`                   | 1 indicates the individual has diabetes, 0 means no diabetes                | Categorical |


In [ ]:
data.describe()

In [ ]:
print("Number of people who have diabetes:", sum(data['Outcome']==1))
print("Number of people who don't have diabetes:", sum(data['Outcome']==0))

## Data quality and preprocessing

Treat zeros in Glucose, BloodPressure, SkinThickness, Insulin, and BMI as missing measurements for this exploratory analysis. Report their counts, then replace them with the nonzero column median without using Outcome. Keep a raw copy. This full-dataset imputation is for EDA only: a future predictive pipeline must fit imputation on training folds. Retain extreme observations rather than automatically altering medically relevant measurements.


In [ ]:
raw_data = data.copy(deep=True)
columns_with_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
print('Zero counts before imputation:')
print(raw_data[columns_with_zeros].eq(0).sum())
data[columns_with_zeros] = data[columns_with_zeros].astype(float).mask(data[columns_with_zeros].eq(0))
data[columns_with_zeros] = data[columns_with_zeros].fillna(data[columns_with_zeros].median())
assert not data[columns_with_zeros].isna().any().any()
assert not data[columns_with_zeros].eq(0).any().any()
assert data['Outcome'].equals(raw_data['Outcome'])
data.describe()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
# Set the size of the plot
plt.figure(figsize=(12, 8))

# Create a boxplot for all columns
sns.boxplot(data=data, orient='h')

# Add labels and title
plt.title('Boxplot of Diabetes Dataset Features')
plt.xlabel('Value')
plt.ylabel('Features')

In [ ]:
def detect_outliers_iqr(df):
    outlier_counts = {}
    
    for column in df.select_dtypes(include=['int64', 'float64']).columns:
        # Calculate Q1 (25th percentile) and Q3 (75th percentile)
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        
        # Determine the lower and upper bounds for outliers
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Count the number of outliers
        outlier_count = df[(df[column] < lower_bound) | (df[column] > upper_bound)].shape[0]
        outlier_counts[column] = outlier_count
    
    return outlier_counts

In [ ]:
detect_outliers_iqr(data)

In [ ]:
# Retain outliers: IQR counts are descriptive flags, not removal criteria.
# No capping is applied to predictors or Outcome.


## Exploratory data analysis

The table below is computed after imputation. Compare it with the raw summary above; no manually maintained statistics are used.


In [ ]:
data.describe()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

data.hist(bins=20, figsize=(12, 10), color='skyblue')
plt.suptitle('Distribution of Numerical Features', fontsize=16)
plt.show()


### Correlation interpretation

These plots describe this sample after imputation. Associations and visual separation do not establish predictive performance, statistical significance, biological mechanisms, or intervention effects. Missingness and preprocessing can affect the apparent relationships.


In [ ]:
plt.figure(figsize=(10, 8))
correlation_matrix = data.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.show()


### Pairplot interpretation

These plots describe this sample after imputation. Associations and visual separation do not establish predictive performance, statistical significance, biological mechanisms, or intervention effects. Missingness and preprocessing can affect the apparent relationships.


In [ ]:
sns.pairplot(data, hue='Outcome', vars=['Glucose', 'BMI', 'Age', 'Insulin', 'DiabetesPedigreeFunction'])
plt.suptitle('Pairplot of Key Features', y=1.02)
plt.show()

#### Count of Each Class in the dataset which on our case "Diabetic", "Non Diabetic"

In [ ]:
sns.countplot(x='Outcome', data=data)
plt.title('Distribution of Outcome (Diabetic vs Non-Diabetic)')
plt.show()

### Distribution interpretation

These plots describe this sample after imputation. Associations and visual separation do not establish predictive performance, statistical significance, biological mechanisms, or intervention effects. Missingness and preprocessing can affect the apparent relationships.


In [ ]:
columns_to_plot = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

# Set the number of subplots
num_columns = len(columns_to_plot)
fig, axes = plt.subplots(nrows=(num_columns + 1) // 2, ncols=2, figsize=(15, 5 * ((num_columns + 1) // 2)))
axes = axes.flatten()  # Flatten the array of axes for easy iteration

# Iterate through each column and create a violin plot
for i, column in enumerate(columns_to_plot):
    sns.violinplot(x='Outcome', y=column, data=data, ax=axes[i])
    axes[i].set_title(f'Violin Plot of {column} by Outcome')

# Adjust layout
plt.tight_layout()
plt.show()

In [ ]:
from itertools import combinations
# List of relevant columns to plot against each other
columns_to_plot = ['Glucose', 'Insulin', 'BMI', 'Age']

# Set up the number of combinations of pairs of columns
column_pairs = list(combinations(columns_to_plot, 2))

# Create a new figure
plt.figure(figsize=(15, 5 * ((len(column_pairs) + 1) // 2)))

# Iterate through each pair of columns and create a scatter plot with trend line
for i, (x_col, y_col) in enumerate(column_pairs):
    plt.subplot((len(column_pairs) + 1) // 2, 2, i + 1)  # Create subplots
    sns.scatterplot(x=x_col, y=y_col, data=data, hue='Outcome', style='Outcome', s=50)
    sns.regplot(x=x_col, y=y_col, data=data, scatter=False, color='gray', line_kws={"color": "red"})
    plt.title(f'Scatter Plot of {x_col} vs {y_col} with Trend Line')
    plt.xlabel(x_col)
    plt.ylabel(y_col)

# Adjust layout
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(x='Pregnancies', hue='Outcome', data=data)
plt.title('Countplot of Pregnancies by Outcome')
plt.show()

In [ ]:
# List of columns to plot
columns_to_plot = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

# Set the number of subplots
num_columns = len(columns_to_plot)
fig, axes = plt.subplots(nrows=(num_columns + 1) // 2, ncols=2, figsize=(15, 5 * ((num_columns + 1) // 2)))
axes = axes.flatten()  # Flatten the array of axes for easy iteration

# Iterate through each column and create a KDE plot
for i, column in enumerate(columns_to_plot):
    sns.kdeplot(data[data['Outcome'] == 0][column], label='Non-Diabetic', fill=True, ax=axes[i], color='blue', alpha=0.5)
    sns.kdeplot(data[data['Outcome'] == 1][column], label='Diabetic', fill=True, ax=axes[i], color='orange', alpha=0.5)
    axes[i].set_title(f'KDE Plot of {column} by Outcome')
    axes[i].legend()

# Adjust layout
plt.tight_layout()
plt.show()

## Results and interpretation

The raw dataset contains 768 records: 500 with Outcome 0 and 268 with Outcome 1. Review the computed correlations and plots for descriptive associations. There is no train/test split, fitted predictive model, or measured predictive performance in this notebook. Earlier numerical interpretation has been removed because preprocessing changed.


## Limitations and next steps

- Confirm the original dataset source, cohort definition, measurement units, target definition, and redistribution license.
- Compare raw and imputed analyses; median imputation can suppress variation, especially for Insulin.
- Do not infer future diabetes onset from a binary recorded outcome without a verified time horizon.
- For prediction, split first and fit all preprocessing inside training folds; compare a dummy classifier and logistic regression using stratified validation.
- Report discrimination, class-specific errors, calibration, and uncertainty before discussing model usefulness.
- Generalization to Egypt or other populations is untested. No clinical use is supported.

See `docs/REVIEW.md` for the prioritized evaluation plan.
